# ROSSA pipeline on a Hugging Face model (Google Colab)

Same ROSSA extraction as before, but the LLM is an **open Hugging Face model running on Colab's GPU** instead of Claude. Nothing runs on my laptop.

Paper: **Doyard 2016**. Model: **Qwen2.5-3B-Instruct** (open, no login, fits the free T4 GPU).

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> **Run all**.

It pulls my prompt files from my GitHub repo, clears out the old Claude results so this is a clean open-model run, generates all three ROSSA sections, then runs my own `process` step to verify every quote and write the review.

In [ ]:
# 1. Install deps (torch is already in Colab). json-repair fixes small JSON slips from smaller models.
!pip -q install -U transformers accelerate bitsandbytes json-repair

In [ ]:
# 2. Get my pipeline + prompt files from GitHub, then clear the old Claude outputs so this run is clean
!rm -rf rossa-llm-pipeline
!git clone -q https://github.com/lUIZhMACEDO/rossa-llm-pipeline
%cd rossa-llm-pipeline

PAPER = "doyard_2016"
SECTIONS = ["study_information", "investigators", "animal_experimentation"]

# keep the prompt files; remove the committed Claude responses/review so they don't leak into this run
!rm -f output/{PAPER}.*.response.json output/{PAPER}.review.md output/{PAPER}.entry.json
print("prompt files kept:")
!ls output/*.prompt.json

In [ ]:
# 3. Load the open model (4-bit so it fits the free T4 GPU)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-3B-Instruct"   # open, no HF login. Bigger option if you have a better GPU: Qwen/Qwen2.5-7B-Instruct

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
print("loaded", MODEL)

In [ ]:
# 4. Run the model on each section and save its JSON reply
import json
from pathlib import Path
from json_repair import repair_json

def parse_json(text):
    # repair_json fixes missing commas, trailing commas, stray prose and ``` fences, then returns valid JSON
    return json.loads(repair_json(text))

def run_section(section):
    prompt = json.loads(Path(f"output/{PAPER}.{section}.prompt.json").read_text())
    messages = [
        {"role": "system", "content": prompt["system_prompt"]},
        {"role": "user",   "content": json.dumps(prompt["request"], ensure_ascii=False)},
    ]
    enc = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=2048, do_sample=False)
    reply = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    obj = parse_json(reply)
    obj["_meta"] = {"backend": "huggingface", "model": MODEL}
    Path(f"output/{PAPER}.{section}.response.json").write_text(json.dumps(obj, indent=2, ensure_ascii=False))
    print(f"[ok] {section}")

for s in SECTIONS:
    print("running", s, "...")
    run_section(s)

In [ ]:
# 5. Run my own pipeline's process step: verify every quote against the paper, write the review
!python rossa_pipeline.py process {PAPER}

# keep a clearly-named copy of this (open-model) run
!cp output/{PAPER}.review.md output/{PAPER}.huggingface.review.md
!cp output/{PAPER}.entry.json output/{PAPER}.huggingface.entry.json

In [ ]:
# 6. Show the result (this open-model run)
from IPython.display import Markdown, display
display(Markdown(Path(f"output/{PAPER}.huggingface.review.md").read_text()))

In [ ]:
# 7. Download the results
from google.colab import files
files.download(f"output/{PAPER}.huggingface.review.md")
files.download(f"output/{PAPER}.huggingface.entry.json")